# TrustHold investigation walkthrough

Run `python -m src.run_project` from the repository root first. This notebook explores observable synthetic activity. The restricted fraud truth file is intentionally not loaded here. An unusual pattern is a lead to investigate, not proof of fraud.

In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd

database = Path('data/synthetic/run/trusthold.sqlite')
connection = sqlite3.connect(database)
transactions = pd.read_sql_query('SELECT * FROM transactions', connection, parse_dates=['Timestamp'])
features = pd.read_sql_query('SELECT * FROM behavior_features', connection, parse_dates=['Timestamp'])
transactions.shape, features.shape

## What changed over time?

Start with transaction count and value by day. This establishes a baseline before exploring who, where, and how activity differs.

In [ ]:
daily = (transactions.assign(Date=transactions['Timestamp'].dt.date)
         .groupby('Date', as_index=False)
         .agg(Transaction_Count=('Transaction_ID', 'size'),
              Transaction_Value=('Amount', 'sum'),
              Average_Amount=('Amount', 'mean')))
daily.tail(15)

## Which terminals connect many customers?

A shared endpoint can be a concentration signal. It may also reflect a busy legitimate merchant, so use it to guide follow-up rather than to label a transaction.

In [ ]:
network = pd.read_sql_query('''
SELECT Terminal_ID, Merchant_ID, Distinct_Customers, Transaction_Count, Total_Amount
FROM terminal_network_summary
ORDER BY Distinct_Customers DESC, Transaction_Count DESC
LIMIT 15''', connection)
network

## What differs from customer baselines?

Amount multiples and prior-24-hour activity are calculated using information available before the current transaction. These features describe behavior; they do not establish intent.

In [ ]:
features.sort_values(['Amount_To_Typical_Spend', 'Tx_Count_Prior_24h'], ascending=False)
        [['Transaction_ID', 'Timestamp', 'Customer_ID', 'Amount',
          'Amount_To_Typical_Spend', 'Tx_Count_Prior_24h',
          'Terminal_Prior_Distinct_Customers', 'Rule_Score', 'Rule_Reasons']]
        .head(25)